# Feature Transformers for Cross-Domain Generalization

This notebook demonstrates the `StructuralFeatures` and `SpectralFeatures`
transformers, which compute fixed-dimensional node features from graph
topology alone.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nocd import NOCD, StructuralFeatures, SpectralFeatures
from nocd.data import load_dataset
from nocd.metrics import overlapping_nmi, evaluate_unsupervised

In [ ]:
graph = load_dataset('data/facebook_ego/fb_698.npz')
A, X, Z_gt = graph['A'], graph['X'], graph['Z']
N, K = Z_gt.shape
print(f'Graph: {N} nodes, {A.nnz} edges, {K} communities')

## StructuralFeatures transformer

Computes 9 topology-derived features per node:
normalized degree, log-degree, clustering coefficient, square clustering,
average neighbor degree, PageRank, HITS hub/authority scores, and core number.

In [ ]:
sf = StructuralFeatures()
X_struct = sf.fit_transform(A)
print(f'Structural features: {X_struct.shape}')
print(f'Sample (node 0): {X_struct[0]}')

In [ ]:
names = ['degree', 'log_deg', 'clust', 'sq_clust',
         'avg_nbr_deg', 'pagerank', 'hub', 'auth', 'core']

fig, axes = plt.subplots(3, 3, figsize=(12, 10))
for i, (ax, name) in enumerate(zip(axes.flat, names)):
    ax.hist(X_struct[:, i], bins=30, alpha=0.7)
    ax.set_title(name)
plt.suptitle('Structural Feature Distributions', fontsize=14)
plt.tight_layout()
plt.show()

## SpectralFeatures transformer

Computes the top-k smallest non-trivial eigenvectors of the normalized
graph Laplacian.

In [ ]:
spf = SpectralFeatures(n_components=16)
X_spec = spf.fit_transform(A)
print(f'Spectral features: {X_spec.shape}')

In [ ]:
z_gt_label = np.argmax(Z_gt, axis=1)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X_spec[:, 0], X_spec[:, 1], c=z_gt_label,
                     cmap='tab20', s=5, alpha=0.7)
ax.set_xlabel('Spectral component 1')
ax.set_ylabel('Spectral component 2')
ax.set_title('Spectral embedding colored by ground-truth community')
plt.colorbar(scatter, label='Community')
plt.tight_layout()
plt.show()

## Comparison across feature types and models

Train all combinations and build a summary table.

In [ ]:
results = []

configs = [
    ('gcn', 'X', dict(hidden_dims=(64,), batch_norm=True)),
    ('gcn', 'structural', dict(hidden_dims=(32, 16), batch_norm=True)),
    ('gcn', 'spectral', dict(hidden_dims=(32, 16), batch_norm=True, n_components=16)),
    ('improved', 'X', dict(hidden_dims=(64,))),
    ('improved', 'structural', dict(hidden_dims=(32, 16))),
    ('improved', 'spectral', dict(hidden_dims=(32, 16), n_components=16)),
]

for model_type, feat_type, kwargs in configs:
    label = f'{model_type} / {feat_type}'
    print(f'Training {label}...')
    m = NOCD(
        num_communities=K,
        model_type=model_type,
        feature_type=feat_type,
        max_epochs=200,
        display_step=200,
        patience=50,
        batch_size=5000,
        **kwargs,
    )
    m.fit(A, X, y=Z_gt, verbose=False)
    Z_pred = m.predict(A, X)
    nmi = overlapping_nmi(Z_pred, Z_gt)
    unsup = evaluate_unsupervised(Z_pred, A)
    results.append({
        'Model': model_type,
        'Features': feat_type,
        'NMI': round(float(nmi), 4),
        'Coverage': round(float(unsup['coverage']), 4),
        'Conductance': round(float(unsup['conductance']), 4),
    })
    print(f'  NMI={nmi:.4f}, Coverage={unsup["coverage"]:.4f}')

In [ ]:
# Display as a table
print(f'{"Model":<12} {"Features":<14} {"NMI":>8} {"Coverage":>10} {"Conductance":>13}')
print('-' * 60)
for r in results:
    print(f'{r["Model"]:<12} {r["Features"]:<14} {r["NMI"]:>8.4f} {r["Coverage"]:>10.4f} {r["Conductance"]:>13.4f}')

In [ ]:
# Plot NMI comparison
models = [f'{r["Model"]}\n{r["Features"]}' for r in results]
nmis = [r['NMI'] for r in results]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2196F3' if 'gcn' in m else '#FF9800' for m in models]
bars = ax.bar(range(len(models)), nmis, color=colors, alpha=0.8)
ax.set_xticks(range(len(models)))
ax.set_xticklabels(models, fontsize=10)
ax.set_ylabel('Overlapping NMI')
ax.set_title('Community Detection: Model x Feature Type')
for bar, nmi in zip(bars, nmis):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{nmi:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## Cross-graph transfer

A model trained with structural features on one graph
can be applied to a different graph.

In [ ]:
# Train on fb_698
m_struct = NOCD(
    num_communities=K,
    model_type='gcn',
    hidden_dims=(32, 16),
    feature_type='structural',
    batch_norm=True,
    max_epochs=200,
    display_step=200,
    patience=50,
    batch_size=5000,
)
m_struct.fit(A, y=Z_gt, verbose=False)

# Apply to fb_348 (different graph)
graph2 = load_dataset('data/facebook_ego/fb_348.npz')
A2, Z_gt2 = graph2['A'], graph2['Z']
print(f'Source graph: {N} nodes, {A.nnz} edges')
print(f'Target graph: {A2.shape[0]} nodes, {A2.nnz} edges')

Z_transfer = m_struct.predict(A2)
unsup = evaluate_unsupervised(Z_transfer, A2)
print(f'\nTransfer unsupervised metrics:')
for k, v in unsup.items():
    print(f'  {k}: {v:.4f}')